# StormEngine V8 — Three-year spatial development

Compare sigma=0.10 and 0.15 under the same reduced 2013–2015 training task and 2016 validation. Both candidates train toward validation convergence (maximum 120 epochs, patience 10); 120 is a safety ceiling, not a required epoch count. These are architecture-development candidates, not final report models.

In [9]:
from pathlib import Path
import subprocess, sys
here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO
DEVICE = 'cuda'
CONFIGS = [
    REPO / 'configs' / 'v8_reconstruction_dev3y_sigma010.yaml',
    REPO / 'configs' / 'v8_reconstruction_dev3y_sigma015.yaml',
]
def run_live(args):
    command = [str(item) for item in args]
    print('Running:', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=REPO, check=True).returncode
ERA5_ROOT = REPO.parent / 'DownloadDate'
FULL_CACHE = ERA5_ROOT / 'cache' / 'stormengine_2010_2017'
DEV_CACHE = ERA5_ROOT / 'cache' / 'stormengine_dev_2013_2016'
print('Repository:', REPO)
print('Development cache:', DEV_CACHE)

Repository: D:\Documents\py_projects\StormEngine-DL\StormEngine-DL
Development cache: D:\Documents\py_projects\StormEngine-DL\DownloadDate\cache\stormengine_dev_2013_2016


In [10]:
# Build once; later spatial/Processor candidates reuse this exact cache.
if not (DEV_CACHE / 'derivation.json').is_file():
    run_live([sys.executable, '-u', REPO / 'scripts' / 'build_development_cache.py',
              'build', '--source-cache', FULL_CACHE, '--output-cache', DEV_CACHE])
run_live([sys.executable, '-u', REPO / 'scripts' / 'build_development_cache.py',
          'verify', '--output-cache', DEV_CACHE])

Running: D:\anaconda3\envs\stormengine\python.exe -u D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\scripts\build_development_cache.py verify --output-cache D:\Documents\py_projects\StormEngine-DL\DownloadDate\cache\stormengine_dev_2013_2016


0

In [11]:
for config in CONFIGS:
    run_live([sys.executable, '-u', REPO / 'scripts' / 'train_v8_reconstruction.py',
              'preflight', '--device', DEVICE, '--config', config])

Running: D:\anaconda3\envs\stormengine\python.exe -u D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\scripts\train_v8_reconstruction.py preflight --device cuda --config D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\configs\v8_reconstruction_dev3y_sigma010.yaml
Running: D:\anaconda3\envs\stormengine\python.exe -u D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\scripts\train_v8_reconstruction.py preflight --device cuda --config D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\configs\v8_reconstruction_dev3y_sigma015.yaml


In [12]:
RUN_THREE_YEAR_CANDIDATES = True
if RUN_THREE_YEAR_CANDIDATES:
    output_dirs = [
        REPO / 'artifacts' / 'v8_spatial_dev3y_converged_sigma010',
        REPO / 'artifacts' / 'v8_spatial_dev3y_converged_sigma015',
    ]
    for config, output in zip(CONFIGS, output_dirs):
        summary = output / 'develop_summary.json'
        if summary.is_file():
            print('Already complete:', output)
            continue
        command = [sys.executable, '-u', REPO / 'scripts' / 'train_v8_reconstruction.py',
                   'develop', '--device', DEVICE, '--config', config]
        checkpoint = output / 'last.pt'
        if checkpoint.is_file():
            command += ['--resume', checkpoint]
        run_live(command)

Running: D:\anaconda3\envs\stormengine\python.exe -u D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\scripts\train_v8_reconstruction.py develop --device cuda --config D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\configs\v8_reconstruction_dev3y_sigma010.yaml


In [13]:
candidates = [
    REPO / 'artifacts' / 'v8_spatial_dev3y_converged_sigma010',
    REPO / 'artifacts' / 'v8_spatial_dev3y_converged_sigma015',
]
if all((path / 'develop_summary.json').is_file() for path in candidates):
    run_live([sys.executable, '-u', REPO / 'scripts' / 'compare_v8_spatial_development.py',
              *candidates, '--output',
              REPO / 'artifacts' / 'v8_spatial_dev3y_comparison.json'])
else:
    print('Run both development candidates before comparison.')

Running: D:\anaconda3\envs\stormengine\python.exe -u D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\scripts\compare_v8_spatial_development.py D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\artifacts\v8_spatial_dev3y_converged_sigma010 D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\artifacts\v8_spatial_dev3y_converged_sigma015 --output D:\Documents\py_projects\StormEngine-DL\StormEngine-DL\artifacts\v8_spatial_dev3y_comparison.json


## Stop here

Send both develop_summary.json files, histories, and v8_spatial_dev3y_comparison.json. The comparison only succeeds after both runs trigger validation early stopping. If either reaches epoch 120 while still improving, raise the common ceiling and rerun both from scratch. Do not start a six-year sigma=0.15 run or Processor training yet.